In [1]:
import os
from openai import OpenAI

In [2]:
def init_model(model_name: str = 'google/gemini-2.0-flash-001'):
    api_key = os.getenv('OPENROUTER_API_KEY')
    base_url = os.getenv('OPENROUTER_BASE_URL', 'https://openrouter.ai/api/v1/chat/completions')
    site_url = os.getenv('OPENROUTER_SITE_URL')
    site_name = os.getenv('OPENROUTER_SITE_NAME')

    client = OpenAI(
        api_key=api_key,
        base_url = base_url,
    )
    completion = client.chat.completions.create(
        extra_hearders={"HTTP-Referer":base_url},
        model=model_name,
        messages=[{
        "role": "user",
        "content": [
        {
          "type": "text",
          "text": "What is in this image?"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "/root/autodl-tmp/ARLP/dataset/test/vlm_query_imgs/apple_agveuv_original.png"
          }
        }
      ]
    }
  ]
)
    print(completion.choices[0].message.content)
    return client, None

In [4]:
import subprocess
import os

In [5]:
env = dict(os.environ)
result = subprocess.run(["clashoff"], shell=True, env=env, capture_output=True, text=True)
print(result.stdout, result.stderr)

 /bin/sh: 1: clashoff: not found



In [2]:
import textwrap
import logging 

logging.basicConfig(
    filename=f'/root/autodl-tmp/ARLP/dataset/test/vlm_responses.log',
    encoding='utf-8',
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

def save_response(cot_resoponse, logger):
    num_question = int(len(cot_resoponse))
    q = 0
    while True:
        raw_sentence = str(cot_resoponse[q])
        wrapped_sentence = textwrap.fill(
            raw_sentence,
            width=80,
            break_long_words=False,
            break_on_hyphens=False,
            subsequent_indent='  '
        )
        if q % 2 == 0:
            logger.info(f'Question {q}:\n{wrapped_sentence}')
        elif q % 2 == 1:
            logger.info(f'Response:\n{wrapped_sentence}')
        q += 1
        if q == num_question:
            break

In [5]:
cot_resoponse1 = ['q1', 'a1', 'q2', 'a2', 'q3', 'a3', '这是一个包含超长词汇的句子supercalifragilisticexpialidocious非常难以阅读需要适当的换行处理。', 'a4']
cot_resoponse2 = ["Let\'s analyze the different colored regions of the baseball bat and determine their functional parts and interaction with people:\n\n1. **Red Region**: This represents the main body or barrel of the bat. It is the part that is primarily used to hit the ball. This part directly interacts with the ball but not with people.\n\n2. **Green Region**: This is likely the transition area or the neck of the bat, which connects the barrel to the handle. This part generally doesn't directly interact with people.\n\n3. **Blue Region**: This represents the handle of the bat, which is where the hands grip the bat. This part directly interacts with the person holding the bat.\n\n4. **Yellow Region**: This section is near the top of the handle, possibly indicating an additional grip area or part of the transition to the knob. It directly interacts with people as it is gripped.\n\n5.", 'a1', 'q2', 'a2', 'q3', 'a3']
save_response(cot_resoponse1, logger)
save_response(cot_resoponse2, logger)

In [2]:
import re
import json
import ast

def parse_region_matching(response_text):
    """
    Parse the structured output from Q4 into a region_matching dict.
    Compatible with pipeline.py's parse_lm_output(parse_dict=True).
    """
    # Try to find ANSWER: marker
    answer_match = re.search(r'ANSWER:\s*(\{.*\})', response_text, re.DOTALL)
    if answer_match:
        dict_str = answer_match.group(1)
    else:
        # Fallback: find any dict-like structure
        dict_match = re.search(r'(\{.*\})', response_text, re.DOTALL)
        if dict_match:
            dict_str = dict_match.group(1)
        else:
            print("Failed to find dict structure in response")
            return None

    # Try ast.literal_eval first (same as pipeline)
    try:
        result = ast.literal_eval(dict_str)
        if isinstance(result, dict):
            return result
    except (ValueError, SyntaxError):
        pass

    # Fallback: try json.loads
    try:
        result = json.loads(dict_str)
        if isinstance(result, dict):
            return result
    except json.JSONDecodeError:
        pass

    print("Failed to parse region matching output:")
    print(response_text)
    return None

In [13]:
response = '''{
  ["Base of the beaker, does not directly interact with people", "Flat and wide, provides stability", "Placement: Designed for sitting flat on surfaces", "Sitting on a flat surface to maintain stability"]: "Red",
  ["Body of the beaker, has indirect interaction", "Forms the primary containment area", "Holding: Lightly grasped if needed", "Observation of contents and measurement"]: "Green",
  ["Main wall, upper mid-section to near the top", "Cylindrical and accessible", "Stirring: Holds and mixes contents", "Visual confirmation and light handling"]: "Blue",
  ["Rim or lip, has direct interaction", "Accessible and precise edge for pouring", "Pouring: Allows precise pouring or filling", "Observation and measurement"]: "Yellow"
}'''
resp2 = "{\"[\\\"The region with affordance is The rim.\\\", \\\"This is the top edge of the beaker.\\\", \\\"The affordance is Pouring. This region allows liquid to be poured into or out of the beaker. The rim guides the flow.\\\", \\\"Further possible affordance:1.Observation. Users can inspect the contents inside through the rim.2.Hold. User can hold the beaker by the rim.\\\"]\" : \"Red\"} "
ANSWER = '''{
    "[\"The region with affordance is The rim.\", \"This is the top edge of the beaker.\", \"The affordance is Pouring. This region allows liquid to be poured into or out of the beaker. The rim guides the flow.\", \"Further possible affordance:1. Observation. Users can inspect the contents inside through the rim.2. Hold. User can hold the beaker by the rim.\" ]": "Yellow",
    "[\"The region with affordance is The upper body.\", \"This area is part of the beaker's main cylindrical body.\", \"The affordance is Gripping. Hands can grasp this area to hold or carry the beaker.\", \"Further possible affordance:1. Observation. Allows visibility of contents. 2. Absorb liquid during transfer.\" ]": "Blue",
    "[\"The region with affordance is The middle body.\", \"This area is part of the beaker’s main cylindrical body.\", \"The affordance is Support Grip. Provides additional support when grasping.\", \"Further possible affordance:1. Holding instruments. Can hold stirrers or other tools.\" ]": "Red",
    "[\"The region with affordance is The base.\", \"This is the flat bottom of the beaker.\", \"The affordance is Placement. Stabilizes the beaker on flat surfaces.\", \"Further possible affordance:1. Measurement. Can be set on calibrated stands. 2. Heat check. May be inspected for temperature.\" ]": "Green"
}'''

In [14]:
data = ast.literal_eval(ANSWER)
print(type(data))
print(data)

SyntaxError: unterminated string literal (detected at line 3) (<unknown>, line 3)